In [ ]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
from typing import List
import time
import os
import numpy as np
import time
import os
import numpy as np
import torch
import pickle
import argparse
from uuid import uuid4

from torch.utils.data import DataLoader

import sys
sys.path.append('..')
sys.path.append('../..')
sys.path.append('../../..')
from oa_reactdiff.trainer.pl_trainer import DDPMModule, ConfModule
from oa_reactdiff.dataset.transition1x import ProcessedTS1x
from oa_reactdiff.analyze.rmsd import batch_rmsd, batch_rmsd_dmae
from oa_reactdiff.analyze.geomopt import calc_deltaE, compute_efh
from oa_reactdiff.evaluate.utils import (
    set_new_schedule,
    inplaint_batch,
    batch_ts_deltaE,
    samples_to_pos_charge
)
from oa_reactdiff.utils.sampling_tools import write_tmp_xyz

EV2KCALMOL = 23.06
AU2KCALMOL = 627.5

In [ ]:
from Model.backbone import generate_backbone
from Model.head import generate_head
from Model.model import MDNet

import yaml
from easydict import EasyDict

In [ ]:
import numpy as np
from ase import Atoms
from ase.calculators.calculator import (Calculator, CalculatorError, 
                    CalculatorSetupError, all_changes, all_properties, kpts2mp, FileIOCalculator)
from pyscf import gto, dft
from pyscf.geomopt.geometric_solver import optimize
from pyscf.hessian import thermo
import pyscf

AU2KCALMOL = 627.509608
AU2EV = 27.2114
BOHR = 0.52917721092

def ase_atoms_to_pyscf(ase_atoms):
    '''Convert ASE atoms to PySCF atom.

    Note: ASE atoms always use A.
    '''
    return [[atom.symbol, atom.position] for atom in ase_atoms]

atoms_from_ase = ase_atoms_to_pyscf

def calculate_efh(
    atom,
    f=True,
    hess=False,
    return_metrics=False,
    xc="wb97x",
    basis="631g*",
    device='gpu',
    # d3=False,
):
    geomfile = ase_atoms_to_pyscf(atom)
    spin = 0
    mol = pyscf.M(
        atom=geomfile,
        unit="Ang",
        basis=basis,
    )
    mol.build()

    if device == 'gpu':
        mf = dft.RKS(mol).to_gpu() if not spin else dft.UKS(mol).to_gpu()
    else:
        mf = dft.RKS(mol) if not spin else dft.UKS(mol)
    mf.xc = xc
    # if d3:
    #     mf = dftd3.dftd3(dft.RKS(mol, xc=xc))
    mf.conv_tol = 1e-6
    # mf.damp = 0.2
    mf.max_cycle = 200
    mf.max_memory = 32000
    mf.run()

    force = None
    force_rms = np.nan
    if mf.converged and f:
        force = mf.nuc_grad_method().kernel() * -1.0 / BOHR
        force_rms = np.sqrt(np.mean(force**2)) * AU2EV
        print("force rms (ev/A): ", force_rms)

    hessian = None
    if hess:
        hessian = mf.Hessian().kernel()
        freq_info = thermo.harmonic_analysis(mf.mol, hessian)
        print("freq: ", freq_info["freq_wavenumber"])

    if return_metrics:
        return (
            mf,
            force,
            hessian,
            force_rms,
            count_negative_eig(freq_info["freq_wavenumber"]),
        )
    return mf, force, hessian

# K-point sampling
def make_kpts(cell, nks):
    raise DeprecationWarning('Use cell.make_kpts(nks) instead.')

def count_negative_eig(x: list):
    count = 0
    for _x in x:
        if _x.imag > 0:
            count += 1
    return count

In [ ]:
parser = argparse.ArgumentParser(description='Training Transition1x dynamics')
parser.add_argument('--config_file', required=True)
parser.add_argument('--log_prefix', default='logs')
parser.add_argument('--notes', default=' ')
parser.add_argument('--device', default='cuda')
parser.add_argument('--resume_status', default=' ')
parser.add_argument('--potential', default=' ')
args = parser.parse_args(['--config_file', "../../../Configs/Potential.yml",
                          '--device', 'cuda',
                          '--potential', '']) # path to trained MLIP checkpoint

In [ ]:
dtype = torch.float32

config_path=args.config_file
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
config = EasyDict(config)
config.notes = args.notes

device = args.device
backbone = generate_backbone(config.model.backbone)
head = generate_head(config.model.head)

In [ ]:
REFERENCE_ENERGIES = {
    1: -13.62222753701504,
    6: -1029.4130839658328,
    7: -1484.8710358098756,
    8: -2041.8396277138045,
    9: -2712.8213146878606,
}

In [ ]:
potential_model = MDNet(backbone, head, REFERENCE_ENERGIES)
best_state = torch.load(args.potential, map_location=device)
potential_model.load_state_dict(best_state['model'])
potential_model.eval()

In [ ]:
potential_model = potential_model.to(device)

In [ ]:
config = dict(
    timesteps=300,
    bz=16,
    resamplings=5,
    jump_length=5,
    repeats=20,
    max_batch=-1,
    shuffle=False,
    single_frag_only=False,
)

In [ ]:
speices = ["reactant", "transition_state", "product"]
keys = ["num_atoms", "charges", "position"]

In [ ]:
print("loading ddpm trainer...")
device = torch.device("cuda")
tspath = "" # path to workdir
checkpoints = {
    "leftnet": f"{tspath}/pretrained-ts1x-diff.ckpt",
}
ddpm_trainer = DDPMModule.load_from_checkpoint(
    checkpoint_path=checkpoints["leftnet"],
    map_location=device,
)
ddpm_trainer = set_new_schedule(
    ddpm_trainer,
    timesteps=config["timesteps"],
    noise_schedule="polynomial_2",
)


print("loading dataset...")
dataset = ProcessedTS1x(
    npz_path="../data/transition1x/test.pkl",
    center=True,
    pad_fragments=0,
    device="cuda",
    zero_charge=False,
    remove_h=False,
    single_frag_only=config["single_frag_only"],
    swapping_react_prod=False,
    use_by_ind=True,
    # ediff=None
    ediff='reactant'
)
loader = DataLoader(
    dataset,
    batch_size=config["bz"],
    shuffle=config["shuffle"],
    num_workers=0,
    collate_fn=dataset.collate_fn,
)

In [ ]:
rmsds, deltaEs, dmaes = [], [], []
positions = []
positions_reactant_product = []
atom_types = []
transition_state_pos_true = []
for num_repeat in range(config["repeats"]):
    rmsds_temp = []
    deltaEs_temp = []
    dmaes_temp = []
    positions_temp = []
    for ii, batch in enumerate(loader):
        print("batch_idx: ", ii)
        time_start = time.time()
        if ii == config["max_batch"]:
            break

        reactant_pos = batch[0][0]['pos']
        transition_state_pos = batch[0][1]['pos']
        product_pos = batch[0][2]['pos']
        atom_type = batch[0][0]['charge'].squeeze()

        batch_num = batch[0][0]['mask']
        ediff_react = batch[1]['ediff'].to(device)

        if num_repeat == 0:
            for j in range(config["bz"]):
                mask = (batch_num == j)
                positions_reactant_product.append((reactant_pos[mask], product_pos[mask]))
                atom_types.append(atom_type[mask])
                transition_state_pos_true.append(transition_state_pos[mask])

        temp_0 = batch[0]
        temp_1 = batch[1]['condition']

        batch = [temp_0, temp_1]
    
        out_samples, xh_fixed, fragments_nodes = inplaint_batch(
            batch,
            ddpm_trainer,
            resamplings=config["resamplings"],
            jump_length=config["jump_length"],
        )

        pred_transition_state_pos = out_samples[1][:,:3]

        energy_pred_react, _ = potential_model.get_energy_and_force(atom_type, reactant_pos, None, None, batch_num)
        energy_pred_trans, __ = potential_model.get_energy_and_force(atom_type, pred_transition_state_pos, None, None, batch_num)
        _rmsds, _dmaes, _aligned_mols = batch_rmsd_dmae(
            fragments_nodes,
            out_samples,
            xh_fixed,
            idx=1,
            threshold=0.5,
        )
        print(_rmsds, _dmaes)

        energy_barrier_pred = energy_pred_react - energy_pred_trans
        _deltaEs = energy_barrier_pred
        _deltaEs = _deltaEs.detach().cpu().numpy()

        print("time cost: ", time.time() - time_start)
        rmsds_temp.append(_rmsds)
        deltaEs_temp.append(_deltaEs)
        dmaes_temp.append(_dmaes)
        positions_temp2 = []
        for j in range(len(_aligned_mols)):
            mask = (batch_num == j)
            atom_reactant = Atoms(numbers=atom_type[mask].cpu().numpy(), positions=reactant_pos[mask].detach().cpu().numpy())
            try:
                atom_trans = Atoms(numbers=atom_type[mask].cpu().numpy(), positions=_aligned_mols[j].cart_coords)
            except:
                atom_trans = Atoms(numbers=atom_type[mask].cpu().numpy(), positions=_aligned_mols[j].cart_coords)
            positions_temp2.append((atom_reactant, atom_trans))
        positions_temp.append(positions_temp2)
    rmsds.append(rmsds_temp)
    deltaEs.append(deltaEs_temp)
    dmaes.append(dmaes_temp)
    positions.append(positions_temp)

In [ ]:
true_energy_barrier = []
for ii, batch in enumerate(loader):
    ediff_react = batch[1]['ediff']
    true_energy_barrier.append(ediff_react.numpy())

In [ ]:
min_rmsds = []
min_dmaes = []
min_barriers_error = []
min_barriers_error_dft = []
transition_state_pos_pred = []

In [ ]:
for i in range(len(rmsds[0])):
    temp_rmsd = [rmsds[j][i] for j in range(len(rmsds))]
    temp_dmae = [dmaes[j][i] for j in range(len(rmsds))]
    temp_deltaE = [deltaEs[j][i] for j in range(len(rmsds))]
    temp_pos = [positions[j][i] for j in range(len(rmsds))]
    temp_rmsd = np.array(temp_rmsd)
    temp_dmae = np.array(temp_dmae)
    temp_deltaE = np.vstack(temp_deltaE)
    min_deltaE_ind = np.argmin(temp_rmsd, axis=0)
    min_rmsd = [temp_rmsd[j][temp] for temp, j in enumerate(min_deltaE_ind)]
    min_dmae = [temp_dmae[j][temp] for temp, j in enumerate(min_deltaE_ind)]
    min_deltaE = [temp_deltaE[j][temp] for temp, j in enumerate(min_deltaE_ind)]
    min_pos = [temp_pos[j][temp] for temp, j in enumerate(min_deltaE_ind)]
    min_rmsds.append(min_rmsd)
    min_dmaes.append(min_dmae)
    min_barriers_error.append(np.abs(min_deltaE - true_energy_barrier[i]))
    temp_dft = []
    for temp in min_pos:
        atom_reactant, atom_trans = temp
        transition_state_pos_pred.append(atom_trans.get_positions())
        # energy_dft_react = calculate_efh(atom_reactant, f=True)[0].e_tot * AU2EV
        # energy_dft_trans = calculate_efh(atom_trans, f=True)[0].e_tot * AU2EV
        # energy_barrier_dft = energy_dft_react - energy_dft_trans
        energy_barrier_dft = 0.0
        temp_dft.append(energy_barrier_dft)
    temp_dft = np.hstack(temp_dft)
    min_barriers_error_dft.append(np.abs(temp_dft - true_energy_barrier[i]))

In [ ]:
import pickle
with open('res_reactdiff.pickle', 'wb') as f:
    pickle.dump({
        'rmsd': np.concatenate(min_rmsds),
        'dmae': np.concatenate(min_dmaes),
        # 'barriers_model': np.concatenate(min_barriers_error),
        # 'barriers_dft': np.concatenate(min_barriers_error_dft)
        'reactant_product_pos': positions_reactant_product,
        'atom_types': atom_types,
        'true_energy_barrier_reactant': np.concatenate(true_energy_barrier),
        'pred_transition_state_pos': transition_state_pos_pred,
        'true_transition_state_pos': transition_state_pos_true
    }, f)